# Pipeline Thu thập, Xử lý và Chuẩn bị Dữ liệu (Wikipedia -> N-gram)

Quá trình này bao gồm phần chính:

**Tokenization & Gán nhãn**: Tách từ tiếng Việt và áp dụng Sliding Window để tạo tập dữ liệu (X, y) cho mô hình N-gram.

## Phần 0: Cài đặt thư viện & Import

In [2]:
!pip install loguru datasets requests pyarrow tqdm underthesea pandas

import hashlib
import html as _html
import json
import os
import re
import time
import unicodedata
import urllib.parse
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from datasets import Dataset, load_dataset
from loguru import logger
from tqdm.auto import tqdm
from underthesea import word_tokenize, sent_tokenize
import logging

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


/home/haloha/.pyenv/versions/3.12.12/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/haloha/.pyenv/versions/3.12.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Mount Google Drive

Mount Drive để lưu trữ dữ liệu lâu dài giữa các session Colab (tránh mất data khi runtime reset).

In [5]:
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/DM'
os.chdir(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Tokenization, Xử lý <UNK>, Split Dataset và Lưu Artifacts

In [5]:
!pip install pyvi pandas scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 20.1 MB/s  0:00:00 15.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pyvi]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import re
import pickle
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Set
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from pyvi import ViTokenizer
import logging

# Cấu hình logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# --- CÁC ĐƯỜNG DẪN CỐ ĐỊNH TỪ PIPELINE TRƯỚC ---
CLEAN_PARQUET_PATH = "/content/drive/MyDrive/DM/data/train/vi_wiki_clean_dedup.parquet"
OUTPUT_DIR = "/content/drive/MyDrive/DM/data/train"

# ==================================================
# PHẦN 1 — START/END TOKEN & TOKENIZATION
# ==================================================
def simple_sent_tokenize(text: str) -> List[str]:
    """
    Tách câu đơn giản bằng Regex (dựa vào dấu chấm, chấm hỏi, chấm than).
    Dùng để thay thế sent_tokenize của underthesea nhằm tối ưu tốc độ.
    """
    # Tách dựa trên dấu kết thúc câu và khoảng trắng theo sau
    sentences = re.split(r'(?<=[.!?])\s+', str(text).strip())
    return [s.strip() for s in sentences if s.strip()]

def tokenize_and_pad(text: str, n_gram: int) -> List[List[str]]:
    """
    Tách câu, làm sạch, tách từ (Word-level bằng PyVi) và thêm padding tokens.
    Số lượng <START> phụ thuộc vào N (N-gram sẽ có N-1 token <START>).
    """
    sentences = simple_sent_tokenize(text)
    padded_sentences = []

    # Số lượng <START> token cần thiết cho cửa sổ N-gram
    n_start = max(1, n_gram - 1)

    for sentence in sentences:
        # Tiền xử lý: Giữ lại chữ cái, số và khoảng trắng, chuyển in thường
        clean_sentence = re.sub(r'[^\w\s]', ' ', sentence.lower())

        # Tách từ bằng PyVi (Từ ghép sẽ được nối bằng dấu '_', VD: 'học_sinh')
        tokenized_string = ViTokenizer.tokenize(clean_sentence)
        tokens = tokenized_string.split()

        if not tokens:
            continue

        # Padding <START> và <END>
        padded = ['<START>'] * n_start + tokens + ['<END>']
        padded_sentences.append(padded)

    return padded_sentences

# ==================================================
# PHẦN 2 — UNKNOWN TOKEN (<UNK>)
# ==================================================
def apply_unk_and_build_vocab(
    corpus: List[List[str]],
    threshold: int
) -> Tuple[List[List[str]], Dict[str, int], Set[str]]:
    """
    Thống kê tần suất từ vựng, chuyển các từ hiếm (freq < threshold) thành <UNK>.
    """
    logger.info("Đang đếm tần suất các từ trong toàn bộ corpus...")
    freq_counter = Counter()
    for sentence in tqdm(corpus, desc="Counting frequencies"):
        freq_counter.update(sentence)

    # Đảm bảo special tokens không bao giờ bị biến thành <UNK>
    special_tokens = {'<START>', '<END>', '<UNK>'}
    for st in special_tokens:
        if st in freq_counter:
            freq_counter[st] = float('inf')

    # Lọc vocabulary (chỉ giữ từ có tần suất >= threshold)
    vocab = {word for word, count in freq_counter.items() if count >= threshold}
    vocab.update(special_tokens)

    logger.info(f"Kích thước Vocabulary gốc: {len(freq_counter):,}")
    logger.info(f"Kích thước Vocabulary sau ngưỡng ({threshold}): {len(vocab):,}")

    # Áp dụng <UNK>
    logger.info("Đang thay thế từ hiếm bằng token <UNK>...")
    processed_corpus = []
    for sentence in tqdm(corpus, desc="Applying <UNK>"):
        new_sentence = [word if word in vocab else '<UNK>' for word in sentence]
        processed_corpus.append(new_sentence)

    # Tính lại dictionary frequency chuẩn xác sau khi gộp <UNK>
    final_freq = Counter()
    for sentence in processed_corpus:
        final_freq.update(sentence)

    return processed_corpus, dict(final_freq), vocab

# ==================================================
# PHẦN 3 — SPLIT DATASET
# ==================================================
def split_dataset(
    corpus: List[List[str]],
    test_ratio: float = 0.1,
    val_ratio: float = 0.1,
    random_state: int = 42
) -> Tuple[List[List[str]], List[List[str]], List[List[str]]]:
    """
    Chia dataset thành Train/Val/Test với tỷ lệ chuẩn (VD: 80/10/10).
    Sử dụng random seed để kết quả tái tạo được.
    """
    logger.info("Đang phân chia Train / Val / Test (80/10/10)...")

    # Tính tỷ lệ validation tương đối trên phần còn lại sau khi tách Test
    relative_val_size = val_ratio / (1.0 - test_ratio)

    train_val, test = train_test_split(
        corpus, test_size=test_ratio, random_state=random_state
    )
    train, val = train_test_split(
        train_val, test_size=relative_val_size, random_state=random_state
    )

    logger.info(f"Kích thước tập Train: {len(train):,} câu")
    logger.info(f"Kích thước tập Val:   {len(val):,} câu")
    logger.info(f"Kích thước tập Test:  {len(test):,} câu")

    return train, val, test

# ==================================================
# PHẦN 4 & 5 — OUTPUT FORMAT & SAVE FILE
# ==================================================
def save_artifacts(
    train: List[List[str]],
    val: List[List[str]],
    test: List[List[str]],
    vocab: Set[str],
    word_freq: Dict[str, int],
    output_dir: str
) -> None:
    """Lưu toàn bộ artifacts ra file pickle (.pkl)."""
    os.makedirs(output_dir, exist_ok=True)

    files_to_save = {
        "train.pkl": train,
        "val.pkl": val,
        "test.pkl": test,
        "vocab.pkl": vocab,
        "word_freq.pkl": word_freq
    }

    for filename, data in files_to_save.items():
        filepath = os.path.join(output_dir, filename)
        logger.info(f"Đang lưu {filename}...")
        with open(filepath, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    logger.info(f"Hoàn tất lưu toàn bộ file tại: {output_dir}")

# ==================================================
# PHẦN 6 — PIPELINE EXECUTION (MAIN)
# ==================================================
def run_corpus_preparation_pipeline(
    parquet_path: str = CLEAN_PARQUET_PATH,
    output_dir: str = OUTPUT_DIR,
    n_gram: int = 3,           # Hỗ trợ N-gram tổng quát (VD: Trigram)
    unk_threshold: int = 2,    # Threshold loại bỏ từ hiếm
    max_docs: int = None       # Tối ưu RAM: Giới hạn số bài viết (nếu cần test)
) -> None:
    """
    Chạy toàn bộ quá trình xử lý Corpus từ file Parquet -> PKL.
    Tối ưu memory bằng cách giải phóng (del) các biến lớn sau khi sử dụng.
    """
    logger.info("=== BẮT ĐẦU PREPROCESSING CORPUS BẰNG PYVI ===")

    # 1. Đọc dữ liệu
    if not os.path.exists(parquet_path):
        logger.error(f"Không tìm thấy file {parquet_path}. Chạy pipeline Crawl trước!")
        return

    logger.info(f"Đọc dữ liệu từ {parquet_path}")
    df = pd.read_parquet(parquet_path)
    if max_docs:
        df = df.head(max_docs)

    # 2. Tokenize & Padding
    corpus_raw = []
    for text in tqdm(df['text'].dropna(), desc="Tokenization & Padding (PyVi)"):
        corpus_raw.extend(tokenize_and_pad(text, n_gram))

    del df # Tối ưu Memory

    # 3. Handle <UNK> & Build Vocab
    corpus_unk, word_freq, vocab = apply_unk_and_build_vocab(corpus_raw, unk_threshold)
    del corpus_raw # Tối ưu Memory

    # 4. Split Train/Val/Test
    train_data, val_data, test_data = split_dataset(corpus_unk)
    del corpus_unk # Tối ưu Memory

    # --- OUTPUT FORMAT THEO YÊU CẦU ---
    logger.info("\n" + "="*50)
    logger.info("DATASET STATISTICS & OUTPUT FORMAT")
    logger.info("="*50)
    logger.info(f"- Tổng số từ vựng (Vocab size): {len(vocab):,}")
    logger.info(f"- Cấu trúc Output (List of Lists): {type(train_data)} chứa {type(train_data[0])}")

    # Print mẫu dữ liệu
    sample_idx = 0
    while len(train_data[sample_idx]) < 5 and sample_idx < len(train_data):
        sample_idx += 1 # Tìm một câu có ý nghĩa để in mẫu

    logger.info(f"- Ví dụ 1 câu sau xử lý: {train_data[sample_idx]}")
    logger.info("="*50 + "\n")

    # 5. Save Artifacts
    save_artifacts(train_data, val_data, test_data, vocab, word_freq, output_dir)
    logger.info("=== PIPELINE HOÀN TẤT THÀNH CÔNG! ===")

# Thực thi Pipeline
if __name__ == "__main__":
    run_corpus_preparation_pipeline(
        n_gram=3,
        unk_threshold=2,
        max_docs=None
    )

2026-05-12 17:06:02,005 - === BẮT ĐẦU PREPROCESSING CORPUS BẰNG PYVI ===
2026-05-12 17:06:02,006 - Đọc dữ liệu từ ./train/vi_wiki_clean_dedup.parquet
Tokenization & Padding (PyVi): 100%|█████| 99972/99972 [09:53<00:00, 168.44it/s]
2026-05-12 17:15:55,759 - Đang đếm tần suất các từ trong toàn bộ corpus...
Counting frequencies: 100%|███████| 2733933/2733933 [00:06<00:00, 422679.98it/s]
2026-05-12 17:16:02,269 - Kích thước Vocabulary gốc: 778,695
2026-05-12 17:16:02,269 - Kích thước Vocabulary sau ngưỡng (2): 328,398
2026-05-12 17:16:02,270 - Đang thay thế từ hiếm bằng token <UNK>...
Applying <UNK>: 100%|█████████████| 2733933/2733933 [00:08<00:00, 304610.90it/s]
2026-05-12 17:16:16,527 - Đang phân chia Train / Val / Test (80/10/10)...
2026-05-12 17:16:17,412 - Kích thước tập Train: 2,187,145 câu
2026-05-12 17:16:17,413 - Kích thước tập Val:   273,394 câu
2026-05-12 17:16:17,413 - Kích thước tập Test:  273,394 câu
2026-05-12 17:16:17,441 - 
2026-05-12 17:16:17,442 - DATASET STATISTICS & O